In [1]:
"""
XL-VLMs Output Visualization
Clean and improved visualization for per-token concept explanations.
"""
import json
import ast
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches

# Set better defaults for matplotlib
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1


In [3]:
# =======================
# Helper Functions
# =======================

def _parse_xyxy_bbox(raw_bbox):
    """Parse bbox in multiple formats into (x1, y1, x2, y2) ints."""
    if raw_bbox is None:
        return None

    def _to_ints(vals):
        try:
            return tuple(int(round(float(v))) for v in vals)
        except Exception:
            return None

    # Already list/tuple
    if isinstance(raw_bbox, (list, tuple)) and len(raw_bbox) == 4:
        return _to_ints(raw_bbox)

    # Dict variants
    if isinstance(raw_bbox, dict):
        if all(k in raw_bbox for k in ("x1", "y1", "x2", "y2")):
            return _to_ints([raw_bbox["x1"], raw_bbox["y1"], raw_bbox["x2"], raw_bbox["y2"]])
        if all(k in raw_bbox for k in ("left", "top", "right", "bottom")):
            return _to_ints([raw_bbox["left"], raw_bbox["top"], raw_bbox["right"], raw_bbox["bottom"]])
        if all(k in raw_bbox for k in ("x", "y", "w", "h")):
            x1 = raw_bbox["x"]; y1 = raw_bbox["y"]
            x2 = x1 + raw_bbox["w"]; y2 = y1 + raw_bbox["h"]
            return _to_ints([x1, y1, x2, y2])
        return None

    # String: try json, then simple split
    if isinstance(raw_bbox, str):
        s = raw_bbox.strip()
        if not s:
            return None
        try:
            parsed = json.loads(s)
            return _parse_xyxy_bbox(parsed)
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(s)
            return _parse_xyxy_bbox(parsed)
        except Exception:
            pass
        for sep in [",", " "]:
            if sep in s:
                parts = [p for p in s.split(sep) if p]
                if len(parts) == 4:
                    return _to_ints(parts)
        return None
    return None


def _crop_to_bbox(img, xyxy):
    """Crop numpy image array to bbox, clipped to image bounds."""
    if xyxy is None:
        return img
    h, w = img.shape[:2]
    x1, y1, x2, y2 = xyxy
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(0, min(x2, w))
    y2 = max(0, min(y2, h))
    if x2 <= x1:
        x2 = min(w, x1 + 1)
    if y2 <= y1:
        y2 = min(h, y1 + 1)
    return img[y1:y2, x1:x2]


def _center_crop_to_max(img, max_size):
    """Center-crop to at most (max_size x max_size) if either dimension > max_size."""
    h, w = img.shape[:2]
    if h <= max_size and w <= max_size:
        return img
    new_h = min(h, max_size)
    new_w = min(w, max_size)
    y1 = max(0, (h - new_h) // 2)
    x1 = max(0, (w - new_w) // 2)
    return img[y1:y1+new_h, x1:x1+new_w]

In [4]:
def visualize_result_per_token(result, idx, save_dir="outputs", max_concepts=3, max_crops=5, concept_size=200):
    """
    Improved vertical layout visualization:
    - Input image at the top
    - Model prediction below it
    - Per-token concept explanations below (one token per row)
    
    Args:
        result: Dict containing visualization data
        idx: Index for filename
        save_dir: Output directory
        max_concepts: Number of top concepts per token to show
        max_crops: Number of crop images per concept
        concept_size: Max size for center-cropping concept images
    """
    os.makedirs(save_dir, exist_ok=True)

    # Load input image
    img_path = result["image_path"]
    img = mpimg.imread(img_path)
    
    ground_truth = result.get("ground_truth", "")
    prediction = result.get("model_output", "")
    n_tokens = len(result.get("per_token_concepts", []))

    # Calculate figure dimensions
    # Rows: 1 (input image) + n_tokens (one per token)
    # Cols: 1 (token label) + max_concepts * (1 bar + max_crops images)
    ncols = 1 + max_concepts * (1 + max_crops)
    nrows = 1 + n_tokens
    
    fig_width = min(24, 4 + ncols * 1.5)
    fig_height = 6 + n_tokens * 2.5
    
    fig = plt.figure(figsize=(fig_width, fig_height))
    
    # Create grid: first row for input image, rest for tokens
    # Width ratios: token label (1.5) + for each concept (bar 0.4 + crops 1 each)
    width_ratios = [1.5]  # Token label column
    for _ in range(max_concepts):
        width_ratios.append(0.4)  # Similarity bar
        width_ratios.extend([1] * max_crops)  # Crop images
    
    gs = fig.add_gridspec(
        nrows, ncols,
        height_ratios=[3] + [1]*n_tokens,
        width_ratios=width_ratios,
        hspace=0.4, wspace=0.1
    )

    # =======================
    # Row 0: Input Image + Prompt + Prediction
    # =======================
    ax_input = fig.add_subplot(gs[0, :])
    ax_input.imshow(img)
    ax_input.axis("off")
    
    # Add prompt as title
    ax_input.set_title("What are items in each grid?", fontsize=14, fontweight='bold', 
                       pad=10, color='darkgreen')
    
    # Add prediction text below the image using ax.text
    prediction_text = f"Prediction: {prediction}"
    if ground_truth:
        match = "✅" if prediction.strip().lower() == ground_truth.strip().lower() else "❌"
        prediction_text = f"GT: {ground_truth} | Pred: {prediction} {match}"
    
    # Position text below the image using axes coordinates
    ax_input.text(0.5, -0.05, prediction_text,
                 transform=ax_input.transAxes,
                 ha='center', va='top', 
                 fontsize=13, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', 
                          edgecolor='orange', linewidth=2))

    # =======================
    # Rows 1+: Per-Token Concepts
    # =======================
    for token_idx, token in enumerate(result.get("per_token_concepts", [])):
        row = 1 + token_idx
        token_text = token.get("token_text", f"Token {token_idx}")
        
        # Column 0: Token label
        ax_token_label = fig.add_subplot(gs[row, 0])
        ax_token_label.axis("off")
        ax_token_label.text(0.5, 0.5, f'"{token_text}"',
                           ha="center", va="center", fontsize=14,
                           fontweight='bold', color="darkred",
                           bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", edgecolor="orange"))

        # Show top concepts for this token
        token_ranks = token.get("top_concepts", [])[:max_concepts]
        
        for concept_idx, concept in enumerate(token_ranks):
            # Parse image grounding paths and bboxes
            paths_str = concept["image_grounding_path"]
            paths = ast.literal_eval(paths_str) if isinstance(paths_str, str) else paths_str
            
            grounding_boxes = None
            gb_raw = concept.get("image_grounding_bboxes", None)
            if gb_raw:
                try:
                    grounding_boxes = json.loads(gb_raw) if isinstance(gb_raw, str) else gb_raw
                except Exception:
                    try:
                        grounding_boxes = ast.literal_eval(gb_raw)
                    except Exception:
                        pass
            
            # Starting column for this concept
            col_start = 1 + concept_idx * (1 + max_crops)
            
            # Similarity bar (vertical - bottom to top)
            ax_bar = fig.add_subplot(gs[row, col_start])
            sim = concept.get("similarity", 0.0)
            ax_bar.bar([0], [sim], color="steelblue", width=0.6)
            ax_bar.set_ylim(0, 1)
            ax_bar.set_xticks([])
            ax_bar.set_yticks([0, 0.5, 1.0])
            ax_bar.tick_params(axis="y", labelsize=8)
            ax_bar.set_xlabel(f"{sim:.3f}", fontsize=10, fontweight='bold')
            
            # Text grounding label above bar
            text_label = ", ".join(concept.get("text_grounding", []))[:40]  # Truncate if long
            ax_bar.set_title(text_label, fontsize=9, color="darkblue", pad=3)
            
            # Concept crop images
            for crop_idx, item in enumerate(paths[:max_crops]):
                try:
                    _, crop_path = item.split("@")
                except ValueError:
                    crop_path = item
                
                ax_crop = fig.add_subplot(gs[row, col_start + 1 + crop_idx])
                ax_crop.axis("off")
                
                if os.path.exists(crop_path):
                    crop_img = mpimg.imread(crop_path)
                    
                    # Apply bounding box crop if available
                    im_bbox = None
                    if grounding_boxes and crop_idx < len(grounding_boxes):
                        im_bbox = grounding_boxes[crop_idx]
                    
                    xyxy = _parse_xyxy_bbox(im_bbox)
                    crop_img = _crop_to_bbox(crop_img, xyxy)
                    crop_img = _center_crop_to_max(crop_img, concept_size)
                    
                    ax_crop.imshow(crop_img)
                else:
                    ax_crop.text(0.5, 0.5, "Missing",
                               ha="center", va="center", fontsize=8, color="red")

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"per_token_viz_{idx}.svg")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved: {save_path}")


In [5]:
def visualize_all_per_token(json_path, save_dir="outputs", max_concepts=3, max_crops=5, concept_size=200):
    """
    Batch visualization: process all results from JSON file.
    
    Args:
        json_path: Path to VLM explanations JSON
        save_dir: Output directory for visualizations
        max_concepts: Top N concepts per token (default: 3)
        max_crops: Number of crop images per concept (default: 5)
        concept_size: Max crop size (default: 200)
    """
    print(f"📂 Loading results from: {json_path}")
    
    with open(json_path, "r") as f:
        data = json.load(f)

    if isinstance(data, dict) and "results" in data:
        results = data["results"]
    elif isinstance(data, list):
        results = data
    else:
        raise ValueError("JSON must be a list or have a 'results' key")

    print(f"📊 Processing {len(results)} results...")
    os.makedirs(save_dir, exist_ok=True)
    
    for idx, result in enumerate(results):
        try:
            visualize_result_per_token(result, idx, save_dir, 
                                      max_concepts=max_concepts, 
                                      max_crops=max_crops, 
                                      concept_size=concept_size)
        except Exception as e:
            print(f"❌ Error processing result {idx}: {e}")
    
    print(f"\n✅ Done! Saved {len(results)} visualizations to: {save_dir}")

In [ ]:
# =======================
# Batch Visualization Function
# =======================
# This function processes all results from a VLM explanations JSON file

In [ ]:
# =======================
# Run Visualization
# =======================

# Use OUTPUT_DIR from .env configuration
# The base OUTPUT_DIR is loaded from .env (e.g., outputs/run_20251016_140704_food)
DECOMP_METHOD = os.environ.get('DECOMP_METHODS', 'snmf').split(',')[0]  # Use first method

# Build paths using environment variables
EXPLANATIONS_JSON = f"{OUTPUT_DIR_BASE}/explanations/{DECOMP_METHOD}/vlm_explanations.json"
VIZ_OUTPUT_DIR = f"{OUTPUT_DIR_BASE}/plots/grounding_per_token"

print(f"\n🎨 Visualization Configuration:")
print(f"  Input JSON: {EXPLANATIONS_JSON}")
print(f"  Output Dir: {VIZ_OUTPUT_DIR}")
print(f"  Method: {DECOMP_METHOD}")

# Check if input file exists
if not os.path.exists(EXPLANATIONS_JSON):
    print(f"\n⚠️  Warning: Input file not found!")
    print(f"  Looking for: {EXPLANATIONS_JSON}")
    print(f"\n💡 To use a different output directory:")
    print(f"  1. Update OUTPUT_DIR in .env file, OR")
    print(f"  2. Manually set paths below:")
    print(f'     EXPLANATIONS_JSON = "/path/to/your/vlm_explanations.json"')
    print(f'     VIZ_OUTPUT_DIR = "/path/to/output/directory"')
else:
    # Run visualization with improved vertical layout
    print(f"\n▶️  Starting visualization...\n")
    visualize_all_per_token(
        json_path=EXPLANATIONS_JSON,
        save_dir=VIZ_OUTPUT_DIR,
        max_concepts=3,      # Top N concepts per token
        max_crops=5,         # Number of crop images per concept
        concept_size=200     # Max size for center-cropping
    )



🎨 Visualization Configuration:
  Input JSON: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/explanations/snmf/vlm_explanations.json
  Output Dir: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token
  Method: snmf

▶️  Starting visualization...

📂 Loading results from: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/explanations/snmf/vlm_explanations.json
📊 Processing 550 results...


/tmp/ipykernel_42623/1813505828.py:154: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_0.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_1.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_2.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_3.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_4.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_5.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_6.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_7.svg
✓ Saved: /mnt/abka03/Projects/xl-vlm-api/xl-vlms/outputs/test1/plots/grounding_per_token/per_token_viz_8.svg
✓ Saved: /mnt/abka0